In [ ]:
%load_ext autoreload
%autoreload 2
from pygsti.modelpacks import smq2Q_XYCNOT as mp2q
from pygsti.tools.leakage import leaky_qubit_model_from_pspec, promote_bb_to_bt
from pygsti.report import construct_standard_report
from pygsti.data import simulate_data
from pygsti.protocols import StandardGST, ProtocolData, ModelEstimateResults
import numpy as np
import scipy.linalg as la

from pygsti.models.explicitmodel import ExplicitOpModel
from pygsti.baseobjs.statespace import ExplicitStateSpace


## GST: modeling a leaky qubit as a qutrit

This short notebook shows how (data from) an experiment design for a two-level system can be used to fit a three-level sytem model, and how to generate a special report to provide insights for these models. The report includes special gate error metrics that reflect the distinguished role of the first two levels in the three-level system.

In [ ]:
edbb = mp2q.create_gst_experiment_design(max_max_length=8)
psbb = mp2q.processor_spec()
tmbb = mp2q.target_model('CPTPLND')
tmbt = promote_bb_to_bt(tmbb)
tmbt.convert_members_inplace('CPTPLND')

In [6]:
def model_with_leaky_gate(m: ExplicitOpModel, gate_label, strength):
    ss = m.state_space
    if not isinstance(ss, ExplicitStateSpace):
        raise NotImplementedError()
    U = np.eye(1, dtype='complex')
    for subsys_udim in ss.tensor_product_blocks_udimensions[0]:
        if subsys_udim == 2:
            u = np.eye(2, dtype='complex')
        else:
            assert subsys_udim == 3
            rng = np.random.default_rng(0)
            v = np.concatenate([[0.0], rng.standard_normal(size=(2,))])
            v /= la.norm(v)
            H = v.reshape((-1, 1)) @ v.reshape((1, -1))
            H *= strength
            u = la.expm(1j*H)
        U = np.kron(U, u)
    m_copy = m.copy()
    G_ideal = m_copy.operations[gate_label]
    from pygsti.modelmembers.operations import ComposedOp, StaticUnitaryOp
    m_copy.operations[gate_label] = ComposedOp([G_ideal, StaticUnitaryOp(U, basis=m.basis)])
    return m_copy, v

In [ ]:
# IMPORTANT: this will run for hours even if you use keyboard interrupts to end
# GST optimization steps early. So ... use those interrupts!

ed = mp2q.create_gst_experiment_design(max_max_length=4)
# ^ Target model. "Leaky" is a bit of a misnomer here. The returned model
#   is simply a qutrit lift of the qubit model; leakage erorrs in the
#   qubit model can manifest as CPTP Markovian errors in the qutrit model.
dgm6, leaking_state = model_with_leaky_gate(tmbt, ('Gxpi2', 0), strength=0.125)
# ^ Data generating model. 
num_samples = 100_000
# ^ The number of samples is large to compensate for short circuit length.
#   Feel free to change the number of samples to something more "realistic"
#   if you'd like.
if num_samples > 10_000:
    from pygsti.objectivefns import objectivefns
    objectivefns.DEFAULT_MIN_PROB_CLIP = objectivefns.DEFAULT_RADIUS = 1e-12
    # ^ There are numerical thresholding rules in objective function evaluation
    #   that lead to errors when the number of samples is extremely large.
    #   The lines above change those thresholding rules to be appropriate in
    #   the unusual setting that is this notebook.
ds = simulate_data(dgm6, ed.all_circuits_needing_data, num_samples=num_samples, seed=1997)
gst = StandardGST(
    modes=('CPTPLND',), target_model=tmbt, verbosity=4,
    badfit_options={'actions': ['wildcard1d'], 'threshold': 0.0}
)
pd = ProtocolData(ed, ds)
res = gst.run(pd)

-- Std Practice:  Iter 1 of 1  (CPTPLND) --: 


/Users/rjmurr/Documents/pygsti-leakage/pygsti-repo/pygsti/modelmembers/povms/__init__.py:615: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  errgen_vec = _np.linalg.lstsq(phys_directions, soln.x)[0]


    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
    Layout for iteration 0
    Num Param Processors (1,)
    MapLayout: 1 processors divided into 1 x 1 (= 1) grid along circuit and parameter directions.
       1 atoms, parameter block size limits (None,)
    *** Distributing 1 atoms to 1 atom-processing groups (1 cores) ***
        More atom-processors than hosts: each host gets ~1 atom-processors
        Atom-processors already occupy a single node, dividing atom-processor into 1 param-processors.
    *** Divided 1-host atom-processor (~1 procs) into 1 param-processing groups ***
    Layout for iteration 1
    Num Param Processors (1,)
    MapLayout: 1 processors divided into 1 x 1 (= 1) grid along circuit and parameter directions.
       1 atoms, parameter block size limits (None,)
    *** Distributing 1 atoms to 1 atom-processing groups (1 cores) ***
        More atom-processors than hosts: each host gets ~1 atom-processors
        Atom-processors already

/Users/rjmurr/Documents/pygsti-leakage/pygsti-repo/pygsti/objectivefns/objectivefns.py:4502: RuntimeWarning: divide by zero encountered in divide
  p5over_lsvec = 0.5/lsvec


      --- Outer Iter 1: norm_f = 47238.7, mu=27167.5, |x|=0.0579551, |J|=48630.8
      --- Outer Iter 2: norm_f = 21747.1, mu=72446.8, |x|=0.077715, |J|=48354.7
      --- Outer Iter 3: norm_f = 13426.5, mu=72603.8, |x|=0.136739, |J|=48345.6
      --- Outer Iter 4: norm_f = 5846.75, mu=71258.8, |x|=0.13757, |J|=48604.3
      --- Outer Iter 5: norm_f = 2836.71, mu=68887.9, |x|=0.138276, |J|=48716.3
      --- Outer Iter 6: norm_f = 1569.43, mu=52991.2, |x|=0.136975, |J|=48121.8
      --- Outer Iter 7: norm_f = 1342.63, mu=51347.5, |x|=0.138124, |J|=47879.7
      --- Outer Iter 8: norm_f = 1285.16, mu=102696, |x|=0.138197, |J|=47725.1
      --- Outer Iter 9: norm_f = 1274.65, mu=130495, |x|=0.138488, |J|=47706.3
      --- Outer Iter 10: norm_f = 1265.7, mu=163903, |x|=0.138718, |J|=47680.6
      --- Outer Iter 11: norm_f = 1257.45, mu=195878, |x|=0.138864, |J|=47644.2
      --- Outer Iter 12: norm_f = 1251.71, mu=242904, |x|=0.138966, |J|=47619.2
      --- Outer Iter 13: norm_f = 1246.88, 

In [ ]:
import os
current_path = os.getcwd()
res.write(current_path + '/example_files/leak_bit_trit')

In [ ]:
rep = construct_standard_report(res, advanced_options={'n_leak': 1})
rep.write_html(current_path + '/example_files/leak_bit_trit_report', auto_open=True)

Running idle tomography
Computing switchable properties


/Users/rjmurr/Documents/pygsti-leakage/pygsti-repo/pygsti/report/factory.py:1252: UserWarning: You should really specify `title=` when generating reports, as this makes it much easier to identify them later on.  Since you didn't, pyGSTi has generated a random one for you: 'fighting friendly grandma'.
  _warnings.warn(("You should really specify `title=` when generating reports,"
